# 05 · CNN: do filtro escrito ao filtro aprendido

Aulas **"CNN: empilhando mapas"**, **"vários filtros"**, **"pooling"**, **"peso"** e **"filtro aprendido"**.
No notebook 02 os kernels eram escritos à mão; no 04 a regra do ICT também. Aqui a rede **aprende**
os próprios filtros a partir dos rótulos.

1. **CNN pequena** em Keras, a receita dos exemplos da aula (convolução 3×3 → pooling, três vezes);
2. com o raio-X **cru** e **pré-processado** (gaussiano + CLAHE, notebook 02);
3. **os filtros que ela aprendeu**: parecem com o Sobel do notebook 02?
4. **mapas de ativação** das três camadas;
5. **transfer learning**: MobileNetV2 treinada no ImageNet, com ajuste fino;
6. **o teste**: CNNs contra o ICT clássico, nos mesmos exames, com bootstrap pareado.

**Aumento de dados sem espelhar.** Os exemplos de imagem costumam espelhar na horizontal. Em raio-X de
tórax isso põe o coração do lado direito, uma anatomia que não existe (fora a rara dextrocardia).
Aqui os aumentos são só pequenas rotações, deslocamentos, zoom e contraste.

O protocolo é o mesmo do notebook 04: parada antecipada e escolha de limiar **na validação**; o teste
só é olhado no fim.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao, cnn
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import filtros

configurar_estilo()
SECAO = "05-cnn"
config.MODELS.mkdir(exist_ok=True)

# True refaz todos os treinos (≈ 1 h no CPU). False reaproveita os modelos já salvos em models/,
# com o histórico gravado ao lado de cada um, e só treina o que faltar.
RETREINAR = False

def treinar_ou_carregar(nome_arquivo, treinar):
    # Carrega models/<nome>.keras e o histórico se existirem (e RETREINAR=False); senão treina.
    arquivo = config.MODELS / f"{nome_arquivo}.keras"
    arquivo_hist = config.MODELS / f"{nome_arquivo}.historico.json"
    if not RETREINAR and arquivo.exists() and arquivo_hist.exists():
        salvo = json.loads(arquivo_hist.read_text())
        print(f"{nome_arquivo}: modelo salvo reaproveitado")
        return keras.models.load_model(arquivo), salvo["historico"], salvo["treino_s"]
    inicio = time.time()
    modelo, historico = treinar()
    tempo = time.time() - inicio
    modelo.save(arquivo)
    arquivo_hist.write_text(json.dumps({"historico": historico, "treino_s": tempo}))
    return modelo, historico, tempo

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
tr, va, te = (conjunto == "treino"), (conjunto == "validacao"), (conjunto == "teste")

X = {"cru": cnn.tensores(imagens), "pré-processado": cnn.tensores(imagens, preprocessar=True)}
print({k: v.shape for k, v in X.items()}, "| treino", tr.sum(), "validação", va.sum(), "teste", te.sum())

def plotar_historico(historicos, titulo, nome):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))
    tabela = {}
    for (rotulo, h), cor in zip(historicos.items(), config.PALETA):
        ax1.plot(h["loss"], color=cor, ls="--", lw=1, label=f"{rotulo} · treino")
        ax1.plot(h["val_loss"], color=cor, lw=2, label=f"{rotulo} · validação")
        ax2.plot(h["auc"], color=cor, ls="--", lw=1)
        ax2.plot(h["val_auc"], color=cor, lw=2)
        for k, v in h.items():
            tabela[f"{rotulo}_{k}"] = pd.Series(v)
    ax1.set(xlabel="época", ylabel="entropia cruzada", title="Perda")
    ax2.set(xlabel="época", ylabel="AUC", title="AUC")
    ax1.legend(fontsize=8)
    fig.suptitle(titulo, fontweight="bold", color=config.AZUL_PROFUNDO)
    salvar_figura(fig, SECAO, nome, pd.DataFrame(tabela))
    plt.show()

## 1. A CNN pequena

Três blocos **convolução 3×3 + ReLU → max pooling 2×2** (16, 32 e 64 filtros). Cada pooling divide a
imagem por 2: 128 → 64 → 32 → 16. No fim, a **média global** de cada um dos 64 mapas vira um vetor de
64 números, que passa por uma camada densa com dropout até a probabilidade de cardiomegalia.

A média global, em vez do `Flatten` dos exemplos da aula, reduz os pesos da parte densa de ~800 mil para
~4 mil: com 3.550 raio-X de treino, menos pesos é menos decoreba.

In [ ]:
keras.utils.set_random_seed(config.SEMENTE)
modelo = cnn.construir_cnn()
modelo.summary()

## 2. Cru ou pré-processado?

A mesma CNN, treinada duas vezes: com o raio-X cru e com o pré-processamento do notebook 02. Até 150
épocas, parando quando a AUC de validação não melhora por 15 épocas e voltando aos melhores pesos.

(Uma primeira rodada com limite de 60 épocas terminou com a AUC de validação ainda subindo e a de treino
quase igual, 0,717 contra 0,710: a rede estava **sub-treinada**, não decorando. Daí o limite maior.)

In [ ]:
modelos, historicos, tempos = {}, {}, {}
for entrada in ["cru", "pré-processado"]:
    def treinar(entrada=entrada):
        keras.utils.set_random_seed(config.SEMENTE)
        m = cnn.construir_cnn()
        h = cnn.treinar(m, X[entrada][tr], y[tr], X[entrada][va], y[va], epocas=150, paciencia=15)
        return m, h.history
    nome = f"CNN pequena · {entrada}"
    modelos[nome], historicos[entrada], tempos[nome] = treinar_ou_carregar(
        f"cnn_pequena_{'crua' if entrada == 'cru' else 'pre'}", treinar)
    h = historicos[entrada]
    print(f"{entrada}: {len(h['loss'])} épocas, melhor AUC de validação {max(h['val_auc']):.3f}, "
          f"{tempos[nome]:.0f} s de treino")
plotar_historico(historicos, "CNN pequena: cru × pré-processado", "historico-cnn-pequena")

## 3. Os filtros que a rede aprendeu

Os 16 kernels 3×3 da primeira camada (`conv1`), lado a lado com os escritos à mão no notebook 02.
Para cada kernel aprendido, a **similaridade de cosseno** com cada kernel à mão (centrado, porque a
média do kernel só muda o brilho): perto de ±1 = mesmo desenho (o sinal só inverte claro e escuro).

In [ ]:
melhor_pequena = max(["CNN pequena · cru", "CNN pequena · pré-processado"],
                     key=lambda k: max(historicos[k.split(" · ")[1]]["val_auc"]))
m = modelos[melhor_pequena]
pesos = m.get_layer("conv1").get_weights()[0][:, :, 0, :]  # (3, 3, 16)

def cosseno(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

a_mao = {"Sobel x": filtros.SOBEL_X, "Sobel y": filtros.SOBEL_Y, "Realce": filtros.REALCE,
         "Gaussiano": filtros.GAUSS_3}
sim = pd.DataFrame([{nome: cosseno(pesos[..., k], kern) for nome, kern in a_mao.items()}
                    for k in range(pesos.shape[-1])])
sim.index = [f"filtro {k + 1}" for k in range(len(sim))]
sim["mais_parecido"] = sim.abs().idxmax(axis=1)
sim["|cosseno|"] = sim[list(a_mao)].abs().max(axis=1)

fig, eixos = plt.subplots(2, 10, figsize=(16, 3.8))
lim = np.abs(pesos).max()
for k in range(16):
    ax = eixos.ravel()[k]
    ax.imshow(pesos[..., k], cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(f"{k + 1}: {sim['mais_parecido'].iloc[k]}\n|cos| {sim['|cosseno|'].iloc[k]:.2f}", fontsize=7)
    ax.axis("off")
for ax, (nome, kern) in zip(eixos.ravel()[16:], a_mao.items()):
    l = np.abs(kern - kern.mean()).max()
    ax.imshow(kern - kern.mean(), cmap="RdBu_r", vmin=-l, vmax=l)
    ax.set_title(f"à mão: {nome}", fontsize=8, color=config.CORAL)
    ax.axis("off")
fig.suptitle(f"conv1 de {melhor_pequena}: filtros aprendidos e os escritos à mão",
             fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "filtros-aprendidos", sim)
plt.show()
sim.round(2).sort_values("|cosseno|", ascending=False).head(8)

## 4. Mapas de ativação

O que cada camada "vê" no raio-X de referência dos notebooks anteriores. A primeira camada responde a
bordas e brilho (como os filtros à mão); as seguintes, já em resolução menor, combinam esses mapas em
padrões maiores.

In [ ]:
entrada_nome = melhor_pequena.split(" · ")[1]
i_ex = int(np.where(catalogo["id"] == "train/true/1298")[0][0])
x_ex = X[entrada_nome][i_ex : i_ex + 1]
extrator = keras.Model(m.inputs, [m.get_layer(f"conv{k}").output for k in (1, 2, 3)])
mapas = extrator.predict(x_ex, verbose=0)

fig, eixos = plt.subplots(3, 9, figsize=(16, 6))
for linha, mapa in enumerate(mapas):
    eixos[linha, 0].imshow(x_ex[0, ..., 0], cmap="gray")
    eixos[linha, 0].set_title("entrada" if linha == 0 else "", fontsize=8)
    eixos[linha, 0].axis("off")
    eixos[linha, 0].text(-10, 64, f"conv{linha + 1}\n{mapa.shape[1]}×{mapa.shape[2]}×{mapa.shape[3]}",
                         ha="right", va="center", fontsize=9, color=config.AZUL_PROFUNDO, fontweight="bold")
    ativos = np.argsort(-mapa[0].mean(axis=(0, 1)))[:8]
    for col, k in enumerate(ativos, start=1):
        eixos[linha, col].imshow(mapa[0, ..., k], cmap="magma")
        eixos[linha, col].set_title(f"mapa {k + 1}", fontsize=7)
        eixos[linha, col].axis("off")
fig.suptitle("Os 8 mapas mais ativos de cada camada", fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "mapas-de-ativacao")
plt.show()

## 5. Transfer learning: MobileNetV2

Em vez de aprender do zero com 3.550 imagens, reaproveitar filtros aprendidos em 1,2 milhão de fotos
do ImageNet. O raio-X tem 1 canal e a MobileNet espera 3: o canal é repetido. Duas fases:

1. **base congelada**, só a cabeça nova aprende (taxa 1e-3);
2. **ajuste fino**: as últimas 30 camadas da base são liberadas, com taxa 100× menor (1e-5), para
   adaptar os filtros sem apagar o que já sabem.

In [ ]:
entrada_mn = "cru"
def treinar_mobilenet():
    keras.utils.set_random_seed(config.SEMENTE)
    mn = cnn.construir_transferencia()
    h1 = cnn.treinar(mn, X[entrada_mn][tr], y[tr], X[entrada_mn][va], y[va], epocas=25, paciencia=6)
    cnn.liberar_topo(mn, n_camadas=30, taxa=1e-5)
    treinaveis = sum(c.trainable for c in next(c for c in mn.layers if c.name.startswith("mobilenet")).layers)
    print(f"camadas da base liberadas para o ajuste fino: {treinaveis}")
    h2 = cnn.treinar(mn, X[entrada_mn][tr], y[tr], X[entrada_mn][va], y[va], epocas=25, paciencia=6)
    return mn, {"fase1": h1.history, "fase2": h2.history}
mobilenet, hist_mn, tempos["MobileNetV2 · ajuste fino"] = treinar_ou_carregar("mobilenet_ajuste_fino", treinar_mobilenet)
modelos["MobileNetV2 · ajuste fino"] = mobilenet
print(f"fase 1: {len(hist_mn['fase1']['loss'])} épocas (melhor AUC val {max(hist_mn['fase1']['val_auc']):.3f}) | "
      f"fase 2: {len(hist_mn['fase2']['loss'])} épocas (melhor AUC val {max(hist_mn['fase2']['val_auc']):.3f}) | "
      f"{tempos['MobileNetV2 · ajuste fino']:.0f} s")
plotar_historico({"base congelada": hist_mn["fase1"], "ajuste fino": hist_mn["fase2"]},
                 "MobileNetV2: cabeça nova e ajuste fino", "historico-mobilenet")

## 6. O teste

Todos os escores nos **mesmos 1.114 exames de teste**: as três redes e o ICT clássico do notebook 04
(lido de `data/processed/ict.csv`). AUC com IC 95 % por bootstrap; limiar de Youden escolhido na
**validação** e aplicado no teste.

In [ ]:
ict = pd.read_csv(config.DATA_PROCESSED / "ict.csv", sep=";").set_index("id").loc[catalogo["id"], "ict"].to_numpy()
escores = {"ICT clássico (notebook 04)": ict}
for nome, mod in modelos.items():
    entrada_nome = "pré-processado" if "pré-processado" in nome else "cru"
    escores[nome] = cnn.prever(mod, X[entrada_nome])

linhas = []
for nome, e in escores.items():
    limiar = avaliacao.limiar_youden(y[va], e[va])
    linhas.append({"modelo": nome, **avaliacao.auc_bootstrap(y[te], e[te]),
                   "auc_validacao": avaliacao.auc_bootstrap(y[va], e[va], n=10)["auc"],
                   **avaliacao.metricas_no_limiar(y[te], e[te], limiar),
                   "treino_s": tempos.get(nome, np.nan)})
resultado = pd.DataFrame(linhas).set_index("modelo").sort_values("auc_validacao", ascending=False)
salvar_tabela(resultado, SECAO, "resultado-teste")
resultado.round(3)

In [ ]:
MELHOR = resultado.drop("ICT clássico (notebook 04)").index[0]  # escolhido pela AUC de validação
print("Melhor rede pela validação:", MELHOR)
pares = {
    f"{MELHOR} − ICT clássico": avaliacao.comparar_auc(y[te], escores[MELHOR][te], ict[te]),
    "CNN pré-processada − CNN crua": avaliacao.comparar_auc(
        y[te], escores["CNN pequena · pré-processado"][te], escores["CNN pequena · cru"][te]),
    "MobileNetV2 − CNN pequena crua": avaliacao.comparar_auc(
        y[te], escores["MobileNetV2 · ajuste fino"][te], escores["CNN pequena · cru"][te]),
}
comparacao = pd.DataFrame(pares).T
salvar_tabela(comparacao, SECAO, "comparacao-pareada")

fig, ax = plt.subplots(figsize=(6.5, 6))
for (nome, e), cor in zip(escores.items(), [config.CORAL, config.AZUL_COBALTO, config.VERDE, config.ROXO]):
    ok = np.isfinite(e[te])
    fpr, tpr, _ = roc_curve(y[te][ok], e[te][ok])
    ax.plot(fpr, tpr, color=cor, lw=2, label=f"{nome} · {resultado.loc[nome, 'auc']:.3f}")
ax.plot([0, 1], [0, 1], color="#999", ls="--", lw=1)
ax.set(xlabel="1 − especificidade", ylabel="sensibilidade", title="Curvas ROC no teste")
ax.legend(loc="lower right", fontsize=8)
salvar_figura(fig, SECAO, "roc-teste")
plt.show()
comparacao.round(3)

In [ ]:
# Escores de todos os exames, para os notebooks 06 e 07.
saida = catalogo[["id", "conjunto", "classe"]].copy()
for nome, e in escores.items():
    saida[nome] = e
saida.to_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";", index=False)
print("Gravado:", config.DATA_PROCESSED / "predicoes.csv")

## Conclusões

* **A CNN pequena supera a régua clássica:** no teste, AUC **0,802** (cru) e **0,809**
  (pré-processado), contra **0,739** do ICT do notebook 04. Na comparação pareada, CNN crua − ICT =
  **+0,064** (IC 95 % 0,030 a 0,097; p = 0,002). Com 27 mil pesos e 3.550 raio-X.
* **Pré-processar não fez diferença:** pré-processado − cru = +0,007 (IC −0,013 a 0,026). A rede
  aprende sozinha o realce de contraste que o CLAHE fazia à mão.
* **A rede reinventou o Sobel:** dos 16 filtros da primeira camada, vários têm similaridade de
  cosseno de 0,84 a 0,90 com o Sobel x ou y escrito à mão no notebook 02. Ninguém disse a ela que
  bordas importam; ela descobriu pelos rótulos.
* **Transfer learning empata, não ganha:** a MobileNetV2 do ImageNet, com ajuste fino das últimas
  camadas, chega a **0,795**; MobileNet − CNN crua = −0,008 (IC −0,035 a 0,020). Com a base congelada
  ela parava em 0,707 na validação; o ajuste fino é que a trouxe ao nível da CNN pequena. Ela tem 80
  vezes mais pesos e mapas finais de só 4×4 a 128 px.
* **O treino ainda não saturou:** a melhor época da CNN crua foi a 150ª, a última, e a da MobileNet
  também foi a última do ajuste fino. Mais épocas provavelmente melhorariam um pouco. Os números aqui
  são um piso, não um teto.
* **Sensibilidade e especificidade dependem do limiar:** com o limiar de Youden da validação, a CNN
  pré-processada fica com 73 % / 72 %; a crua, com 57 % / 83 %. A AUC é parecida, o ponto de
  operação não: para triagem, o limiar precisa ser escolhido pensando no custo de cada erro.

> **Nota de método.** Uma primeira versão deste notebook tinha um erro: o ajuste fino liberava a
> camada de aumento de dados em vez da MobileNet (as duas são modelos aninhados no Keras), e a
> MobileNet ficou em 0,712. O erro foi corrigido e testado (`tests/test_modelos.py`); as CNNs
> pequenas não eram afetadas e foram reaproveitadas (`RETREINAR = False`).

**O que ainda não se sabe:** a AUC de 0,80 diz que a rede separa as classes, não **como**. O notebook
06 testa se ela precisa ver o coração e o 07 mostra para onde ela olha.